# 13 — CNN-LSTM (Optional; Deferred)

This experiment is intentionally not implemented in the core pipeline. The current environment does not include PyTorch, and only 126 patients are represented, with multiple diagnoses present in very few patients. Do not use cycle-level random splitting to make this experiment appear larger.

If resources and an appropriate patient-level experiment become available, implement a temporal model as a distinct extension and evaluate it using the same patient split and unseen-test discipline. The project is not blocked by this optional stage.

In [1]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1


In [2]:
PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

DL_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
)

CNN_BASELINE_DIR = (
    DL_DIR
    / "cnn_baseline"
)

CNN_LSTM_OUTPUT_DIR = (
    DL_DIR
    / "cnn_lstm"
)

CNN_LSTM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("CNN-LSTM output directory:")
print(CNN_LSTM_OUTPUT_DIR)

CNN-LSTM output directory:
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: mps


In [4]:
TRAIN_MANIFEST_PATH = (
    CNN_BASELINE_DIR
    / "train_cnn_manifest.csv"
)

VAL_MANIFEST_PATH = (
    CNN_BASELINE_DIR
    / "validation_cnn_manifest.csv"
)

TEST_MANIFEST_PATH = (
    CNN_BASELINE_DIR
    / "test_cnn_manifest.csv"
)

train_manifest = pd.read_csv(
    TRAIN_MANIFEST_PATH
)

val_manifest = pd.read_csv(
    VAL_MANIFEST_PATH
)

test_manifest = pd.read_csv(
    TEST_MANIFEST_PATH
)

print("Train:", train_manifest.shape)
print("Validation:", val_manifest.shape)
print("Test:", test_manifest.shape)

Train: (4423, 8)
Validation: (914, 8)
Test: (1561, 8)


In [5]:
train_patients = set(
    train_manifest["patient_id"].unique()
)

val_patients = set(
    val_manifest["patient_id"].unique()
)

test_patients = set(
    test_manifest["patient_id"].unique()
)

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

print(
    "Train/Val overlap:",
    train_patients & val_patients
)

print(
    "Train/Test overlap:",
    train_patients & test_patients
)

print(
    "Val/Test overlap:",
    val_patients & test_patients
)

Train patients: 82
Validation patients: 19
Test patients: 25
Train/Val overlap: set()
Train/Test overlap: set()
Val/Test overlap: set()


In [6]:
CLASS_MAPPING_PATH = (
    DL_DIR
    / "disease_class_mapping.csv"
)

class_mapping_df = pd.read_csv(
    CLASS_MAPPING_PATH
)

display(class_mapping_df)

,disease_label,diagnosis
0,0,Asthma
1,1,Bronchiectasis
2,2,Bronchiolitis
3,3,COPD
4,4,Healthy
5,5,LRTI
6,6,Pneumonia
7,7,URTI


In [8]:
print("Class mapping columns:")
print(class_mapping_df.columns.tolist())

print("\nClass mapping:")
display(class_mapping_df)

Class mapping columns:
['disease_label', 'diagnosis']

Class mapping:


,disease_label,diagnosis
0,0,Asthma
1,1,Bronchiectasis
2,2,Bronchiolitis
3,3,COPD
4,4,Healthy
5,5,LRTI
6,6,Pneumonia
7,7,URTI


In [9]:
class_names = (
    class_mapping_df
    .sort_values("disease_label")["diagnosis"]
    .tolist()
)

num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

Classes: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Number of classes: 8


In [10]:
TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
FMIN = 0
FMAX = 2000

print("Target sample rate:", TARGET_SR)
print("Target duration:", TARGET_DURATION)
print("Target samples:", TARGET_SAMPLES)
print("Log-Mel configuration:")
print(f"  N_FFT: {N_FFT}")
print(f"  Hop length: {HOP_LENGTH}")
print(f"  Mel bands: {N_MELS}")
print(f"  Frequency range: {FMIN}-{FMAX} Hz")

Target sample rate: 4000
Target duration: 5.0
Target samples: 20000
Log-Mel configuration:
  N_FFT: 512
  Hop length: 128
  Mel bands: 64
  Frequency range: 0-2000 Hz


In [11]:
NORMALIZATION_PATH = (
    DL_DIR / "logmel_train_normalization.csv"
)

normalization_df = pd.read_csv(
    NORMALIZATION_PATH
)

print("Normalization statistics:")
display(normalization_df)

Normalization statistics:


,statistic,value
0,mean,-66.740814
1,std,17.337090


In [12]:
print("Normalization columns:")
print(normalization_df.columns.tolist())

Normalization columns:
['statistic', 'value']


In [13]:
print("Normalization statistics:")
display(normalization_df)

Normalization statistics:


,statistic,value
0,mean,-66.740814
1,std,17.337090


In [14]:
logmel_mean = float(
    normalization_df.loc[
        normalization_df["statistic"] == "mean",
        "value"
    ].iloc[0]
)

logmel_std = float(
    normalization_df.loc[
        normalization_df["statistic"] == "std",
        "value"
    ].iloc[0]
)

print("Log-Mel training mean:", logmel_mean)
print("Log-Mel training std:", logmel_std)

assert np.isfinite(logmel_mean)
assert np.isfinite(logmel_std)
assert logmel_std > 0

Log-Mel training mean: -66.74081420898438
Log-Mel training std: 17.33708953857422


In [15]:
import soundfile as sf
import librosa

In [16]:
print("Train manifest columns:")
print(train_manifest.columns.tolist())

print("\nFirst row:")
display(train_manifest.head(1))

Train manifest columns:
['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']

First row:


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path,cnn_split
0,102,1b1,Ar,Healthy,4,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,train


In [18]:
class CNNLSTMRespiratoryDataset(Dataset):

    def __init__(
        self,
        manifest_df,
        mean,
        std,
        augment=False
    ):
        self.df = manifest_df.reset_index(drop=True)
        self.mean = mean
        self.std = std
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        # Load processed 4 kHz audio
        signal = load_processed_audio(
            row["audio_path"]
        )

        # Augmentation only for training
        if self.augment:
            signal = augment_training_audio(
                signal
            )

        # Extract Log-Mel representation
        logmel = extract_logmel(
            signal
        )

        # Training-only normalization statistics
        logmel = (
            logmel - self.mean
        ) / self.std

        # Convert to tensor
        logmel = torch.tensor(
            logmel,
            dtype=torch.float32
        )

        # CNN expects:
        # [channel, mel_bins, time]
        logmel = logmel.unsqueeze(0)

        # Disease label
        label = torch.tensor(
            int(row["disease_label"]),
            dtype=torch.long
        )

        return logmel, label

In [19]:
train_dataset_lstm = CNNLSTMRespiratoryDataset(
    train_manifest,
    logmel_mean,
    logmel_std,
    augment=True
)

val_dataset_lstm = CNNLSTMRespiratoryDataset(
    val_manifest,
    logmel_mean,
    logmel_std,
    augment=False
)

test_dataset_lstm = CNNLSTMRespiratoryDataset(
    test_manifest,
    logmel_mean,
    logmel_std,
    augment=False
)

print("Train samples:", len(train_dataset_lstm))
print("Validation samples:", len(val_dataset_lstm))
print("Test samples:", len(test_dataset_lstm))

Train samples: 4423
Validation samples: 914
Test samples: 1561


In [21]:
def fix_audio_length(signal, target_samples):
    if len(signal) < target_samples:
        signal = np.pad(
            signal,
            (0, target_samples - len(signal)),
            mode="constant"
        )
    else:
        signal = signal[:target_samples]

    return signal.astype(np.float32)


def load_processed_audio(audio_path):
    signal, sr = sf.read(audio_path)

    if signal.ndim > 1:
        signal = np.mean(signal, axis=1)

    if sr != TARGET_SR:
        raise ValueError(
            f"Expected {TARGET_SR} Hz, got {sr} Hz"
        )

    signal = signal.astype(np.float32)

    return fix_audio_length(
        signal,
        TARGET_SAMPLES
    )

In [23]:
import soundfile as sf
import librosa

In [25]:
def add_gaussian_noise(signal, noise_factor=0.005):
    noise = np.random.normal(
        0,
        1,
        size=signal.shape
    ).astype(np.float32)

    return (
        signal + noise_factor * noise
    ).astype(np.float32)


def random_time_shift(signal, max_shift_ratio=0.10):
    max_shift = int(len(signal) * max_shift_ratio)

    if max_shift == 0:
        return signal.copy()

    shift = np.random.randint(
        -max_shift,
        max_shift + 1
    )

    if shift == 0:
        return signal.copy()

    shifted = np.zeros_like(signal)

    if shift > 0:
        shifted[shift:] = signal[:-shift]
    else:
        shifted[:shift] = signal[-shift:]

    return shifted


def augment_training_audio(signal):
    signal = random_time_shift(
        signal,
        max_shift_ratio=0.10
    )

    signal = add_gaussian_noise(
        signal,
        noise_factor=0.005
    )

    return signal.astype(np.float32)

In [27]:
def extract_logmel(signal):
    mel_power = librosa.feature.melspectrogram(
        y=signal,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0
    )

    mel_db = librosa.power_to_db(
        mel_power,
        ref=np.max
    )

    return mel_db.astype(np.float32)

In [28]:
sample_x, sample_y = train_dataset_lstm[0]

print("Input shape:", sample_x.shape)
print("Label:", sample_y.item())
print("Input dtype:", sample_x.dtype)
print("Input finite:", torch.isfinite(sample_x).all().item())
print(
    "Input range:",
    sample_x.min().item(),
    "to",
    sample_x.max().item()
)

Input shape: torch.Size([1, 64, 157])
Label: 4
Input dtype: torch.float32
Input finite: True
Input range: -0.21601104736328125 to 3.849597454071045


In [29]:
from sklearn.utils.class_weight import compute_class_weight

train_labels_lstm = (
    train_manifest["disease_label"]
    .astype(int)
    .to_numpy()
)

class_weights_lstm = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_labels_lstm
)

class_weights_lstm = torch.tensor(
    class_weights_lstm,
    dtype=torch.float32,
    device=device
)

print("Class weights:")
for name, weight in zip(class_names, class_weights_lstm.cpu().numpy()):
    print(f"{name}: {weight:.4f}")

Class weights:
Asthma: 92.1458
Bronchiectasis: 7.2747
Bronchiolitis: 4.8498
COPD: 0.1489
Healthy: 2.8499
LRTI: 17.2773
Pneumonia: 4.5692
URTI: 3.3106


In [30]:
class_weights_df = pd.DataFrame({
    "disease_label": np.arange(num_classes),
    "diagnosis": class_names,
    "class_weight": class_weights_lstm.cpu().numpy()
})

class_weights_path = (
    CNN_LSTM_OUTPUT_DIR /
    "cnn_lstm_training_class_weights.csv"
)

class_weights_df.to_csv(
    class_weights_path,
    index=False
)

print("Saved:", class_weights_path)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/cnn_lstm_training_class_weights.csv


In [31]:
import torch.nn as nn


class CNNLSTM(nn.Module):
    def __init__(
        self,
        num_classes,
        lstm_hidden_size=64,
        lstm_layers=2
    ):
        super().__init__()

        # CNN feature extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(
                1, 16,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=(2, 1)
            ),

            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=(2, 1)
            ),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        # Temporal sequence model
        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=0.30 if lstm_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(0.30)

        self.classifier = nn.Linear(
            lstm_hidden_size,
            num_classes
        )

    def forward(self, x):

        # x: [B, 1, 64, 157]
        x = self.cnn(x)

        # [B, 64, 16, 157]
        # Average across frequency/mel dimension
        x = x.mean(dim=2)

        # [B, 64, 157]
        # Convert to [B, time, features]
        x = x.transpose(1, 2)

        # [B, 157, 64]
        lstm_out, _ = self.lstm(x)

        # Aggregate temporal information
        x = lstm_out.mean(dim=1)

        x = self.dropout(x)

        return self.classifier(x)

In [32]:
model_lstm = CNNLSTM(
    num_classes=num_classes,
    lstm_hidden_size=64,
    lstm_layers=2
).to(device)

print(model_lstm)

CNNLSTM(
  (cnn): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
  )
  (lstm): LSTM(64, 64, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=64, out_features=8, bias=True)
)


In [34]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader_lstm = DataLoader(
    train_dataset_lstm,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader_lstm = DataLoader(
    val_dataset_lstm,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader_lstm = DataLoader(
    test_dataset_lstm,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("Train batches:", len(train_loader_lstm))
print("Validation batches:", len(val_loader_lstm))
print("Test batches:", len(test_loader_lstm))

Train batches: 139
Validation batches: 29
Test batches: 49


In [35]:
with torch.no_grad():
    sample_batch_x, sample_batch_y = next(
        iter(train_loader_lstm)
    )

    sample_batch_x = sample_batch_x.to(device)

    sample_output = model_lstm(sample_batch_x)

print("Input shape :", sample_batch_x.shape)
print("Output shape:", sample_output.shape)
print("Expected    :", (sample_batch_x.shape[0], num_classes))

Input shape : torch.Size([32, 1, 64, 157])
Output shape: torch.Size([32, 8])
Expected    : (32, 8)


In [36]:
import torch.optim as optim

criterion_lstm = nn.CrossEntropyLoss(
    weight=class_weights_lstm
)

optimizer_lstm = optim.Adam(
    model_lstm.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler_lstm = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_lstm,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

print("Loss:", criterion_lstm)
print("Initial learning rate:", optimizer_lstm.param_groups[0]["lr"])

Loss: CrossEntropyLoss()
Initial learning rate: 0.001


In [37]:
def train_one_epoch_lstm(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [38]:
def evaluate_lstm(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for inputs, labels in loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item() * labels.size(0)
            )

            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return (
        epoch_loss,
        epoch_accuracy,
        np.array(all_labels),
        np.array(all_predictions)
    )

In [40]:
MAX_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 7

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

history_lstm = []

best_model_path = (
    CNN_LSTM_OUTPUT_DIR /
    "best_cnn_lstm.pt"
)

print("Max epochs:", MAX_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Best model:", best_model_path)

Max epochs: 40
Early stopping patience: 7
Best model: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/best_cnn_lstm.pt


In [41]:
for epoch in range(1, MAX_EPOCHS + 1):

    train_loss, train_accuracy = train_one_epoch_lstm(
        model_lstm,
        train_loader_lstm,
        criterion_lstm,
        optimizer_lstm,
        device
    )

    (
        val_loss,
        val_accuracy,
        _,
        _
    ) = evaluate_lstm(
        model_lstm,
        val_loader_lstm,
        criterion_lstm,
        device
    )

    scheduler_lstm.step(val_loss)

    current_lr = optimizer_lstm.param_groups[0]["lr"]

    history_lstm.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "learning_rate": current_lr
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            model_lstm.state_dict(),
            best_model_path
        )

        print(
            f"  -> Best model saved "
            f"(validation loss = {val_loss:.4f})"
        )

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        break

Epoch 01/40 | Train Loss: 1.9095 | Train Acc: 0.4924 | Val Loss: 1.8143 | Val Acc: 0.3523 | LR: 0.001000
  -> Best model saved (validation loss = 1.8143)
Epoch 02/40 | Train Loss: 1.6961 | Train Acc: 0.4755 | Val Loss: 1.8661 | Val Acc: 0.3140 | LR: 0.001000
Epoch 03/40 | Train Loss: 1.6477 | Train Acc: 0.4691 | Val Loss: 1.4393 | Val Acc: 0.5941 | LR: 0.001000
  -> Best model saved (validation loss = 1.4393)
Epoch 04/40 | Train Loss: 1.6244 | Train Acc: 0.4260 | Val Loss: 1.3183 | Val Acc: 0.5667 | LR: 0.001000
  -> Best model saved (validation loss = 1.3183)
Epoch 05/40 | Train Loss: 1.5102 | Train Acc: 0.5333 | Val Loss: 1.5513 | Val Acc: 0.6324 | LR: 0.001000
Epoch 06/40 | Train Loss: 1.4429 | Train Acc: 0.4712 | Val Loss: 1.8704 | Val Acc: 0.6980 | LR: 0.001000
Epoch 07/40 | Train Loss: 1.4028 | Train Acc: 0.5261 | Val Loss: 1.4009 | Val Acc: 0.6488 | LR: 0.001000
Epoch 08/40 | Train Loss: 1.3833 | Train Acc: 0.5200 | Val Loss: 1.2719 | Val Acc: 0.7374 | LR: 0.001000
  -> Best mod

In [42]:
history_lstm_df = pd.DataFrame(history_lstm)

history_path = (
    CNN_LSTM_OUTPUT_DIR /
    "cnn_lstm_training_history.csv"
)

history_lstm_df.to_csv(
    history_path,
    index=False
)

print("Saved:", history_path)
print()
print(history_lstm_df)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/cnn_lstm_training_history.csv

    epoch  train_loss  train_accuracy  val_loss  val_accuracy  learning_rate
0       1    1.909523        0.492426  1.814313      0.352298         0.0010
1       2    1.696076        0.475469  1.866099      0.314004         0.0010
2       3    1.647651        0.469139  1.439280      0.594092         0.0010
3       4    1.624387        0.425955  1.318295      0.566740         0.0010
4       5    1.510170        0.533348  1.551295      0.632385         0.0010
5       6    1.442926        0.471173  1.870385      0.698031         0.0010
6       7    1.402817        0.526113  1.400892      0.648796         0.0010
7       8    1.383328        0.520009  1.271949      0.737418         0.0010
8       9    1.325254        0.539227  1.422153      0.748359         0.0010
9      10    1.370693        0.553244  1.323049      0.738512         0.0010
10     11    1.314444     

In [43]:
training_config_lstm = {
    "model": "CNN-LSTM",
    "target_sample_rate": TARGET_SR,
    "target_duration_seconds": TARGET_DURATION,
    "target_samples": TARGET_SAMPLES,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "n_mels": N_MELS,
    "fmin": FMIN,
    "fmax": FMAX,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "scheduler_factor": 0.5,
    "scheduler_patience": 3,
    "scheduler_min_lr": 1e-6,
    "lstm_hidden_size": 64,
    "lstm_layers": 2,
    "lstm_dropout": 0.30,
    "classifier_dropout": 0.30,
    "augmentation": "training_only_noise_and_time_shift",
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss
}

config_path = (
    CNN_LSTM_OUTPUT_DIR /
    "cnn_lstm_training_config.json"
)

with open(config_path, "w") as f:
    json.dump(
        training_config_lstm,
        f,
        indent=2
    )

print("Saved:", config_path)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/cnn_lstm_training_config.json


In [44]:
model_lstm.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

model_lstm.to(device)

print("Best model loaded.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

Best model loaded.
Best epoch: 8
Best validation loss: 1.271948614057879


In [45]:
(
    test_loss_lstm,
    test_accuracy_lstm,
    test_labels_lstm,
    test_predictions_lstm
) = evaluate_lstm(
    model_lstm,
    test_loader_lstm,
    criterion_lstm,
    device
)

print("Test loss:", test_loss_lstm)
print("Test accuracy:", test_accuracy_lstm)
print("Test samples:", len(test_labels_lstm))

Test loss: 0.8616819239670768
Test accuracy: 0.8135810377962844
Test samples: 1561


In [46]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy_lstm = accuracy_score(
    test_labels_lstm,
    test_predictions_lstm
)

balanced_accuracy_lstm = balanced_accuracy_score(
    test_labels_lstm,
    test_predictions_lstm
)

macro_precision_lstm = precision_score(
    test_labels_lstm,
    test_predictions_lstm,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

macro_recall_lstm = recall_score(
    test_labels_lstm,
    test_predictions_lstm,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

macro_f1_lstm = f1_score(
    test_labels_lstm,
    test_predictions_lstm,
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
)

weighted_f1_lstm = f1_score(
    test_labels_lstm,
    test_predictions_lstm,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:          {accuracy_lstm:.6f}")
print(f"Balanced Accuracy: {balanced_accuracy_lstm:.6f}")
print(f"Macro Precision:   {macro_precision_lstm:.6f}")
print(f"Macro Recall:      {macro_recall_lstm:.6f}")
print(f"Macro F1:          {macro_f1_lstm:.6f}")
print(f"Weighted F1:       {weighted_f1_lstm:.6f}")

Accuracy:          0.813581
Balanced Accuracy: 0.226085
Macro Precision:   0.134473
Macro Recall:      0.169564
Macro F1:          0.146471
Weighted F1:       0.786735


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [47]:
report_lstm = classification_report(
    test_labels_lstm,
    test_predictions_lstm,
    labels=np.arange(num_classes),
    target_names=class_names,
    zero_division=0
)

print(report_lstm)

                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         0
Bronchiectasis       0.00      0.00      0.00        20
 Bronchiolitis       0.00      0.00      0.00        19
          COPD       0.88      0.93      0.91      1337
       Healthy       0.00      0.00      0.00        84
          LRTI       0.00      0.00      0.00         0
     Pneumonia       0.19      0.42      0.26        52
          URTI       0.00      0.00      0.00        49

      accuracy                           0.81      1561
     macro avg       0.13      0.17      0.15      1561
  weighted avg       0.76      0.81      0.79      1561



In [48]:
cm_lstm = confusion_matrix(
    test_labels_lstm,
    test_predictions_lstm,
    labels=np.arange(num_classes)
)

cm_lstm_df = pd.DataFrame(
    cm_lstm,
    index=class_names,
    columns=class_names
)

print(cm_lstm_df)

                Asthma  Bronchiectasis  Bronchiolitis  COPD  Healthy  LRTI  \
Asthma               0               0              0     0        0     0   
Bronchiectasis       0               0              0    20        0     0   
Bronchiolitis        0               0              0    17        0     0   
COPD                24               0              0  1248        0     0   
Healthy              7               0              0    60        0     0   
LRTI                 0               0              0     0        0     0   
Pneumonia            4               0              0    26        0     0   
URTI                 0               0              0    40        0     0   

                Pneumonia  URTI  
Asthma                  0     0  
Bronchiectasis          0     0  
Bronchiolitis           2     0  
COPD                   65     0  
Healthy                17     0  
LRTI                    0     0  
Pneumonia              22     0  
URTI                    9

In [49]:
metrics_lstm_df = pd.DataFrame([{
    "model": "CNN-LSTM",
    "accuracy": accuracy_lstm,
    "balanced_accuracy": balanced_accuracy_lstm,
    "macro_precision": macro_precision_lstm,
    "macro_recall": macro_recall_lstm,
    "macro_f1": macro_f1_lstm,
    "weighted_f1": weighted_f1_lstm,
    "test_loss": test_loss_lstm,
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss,
    "test_samples": len(test_labels_lstm),
    "test_patients": test_manifest["patient_id"].nunique()
}])

metrics_path = (
    CNN_LSTM_OUTPUT_DIR /
    "cnn_lstm_metrics.csv"
)

metrics_lstm_df.to_csv(
    metrics_path,
    index=False
)

print("Saved:", metrics_path)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/cnn_lstm_metrics.csv


In [50]:
cm_path = (
    CNN_LSTM_OUTPUT_DIR /
    "cnn_lstm_confusion_matrix.csv"
)

cm_lstm_df.to_csv(cm_path)

predictions_lstm_df = test_manifest.copy()

predictions_lstm_df["predicted_disease_label"] = (
    test_predictions_lstm
)

predictions_lstm_df["predicted_diagnosis"] = [
    class_names[i]
    for i in test_predictions_lstm
]

predictions_path = (
    CNN_LSTM_OUTPUT_DIR /
    "cnn_lstm_test_predictions.csv"
)

predictions_lstm_df.to_csv(
    predictions_path,
    index=False
)

print("Saved:", cm_path)
print("Saved:", predictions_path)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/cnn_lstm_confusion_matrix.csv
Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/cnn_lstm_test_predictions.csv


In [51]:
cnn_baseline_metrics_path = (
    DL_DIR /
    "cnn_baseline" /
    "cnn_baseline_metrics.csv"
)

multifeature_a_metrics_path = (
    DL_DIR /
    "multifeature_cnn" /
    "multifeature_cnn_metrics.csv"
)

multifeature_b_metrics_path = (
    DL_DIR /
    "multifeature_cnn_12b" /
    "metrics.csv"
)

comparison_rows = []

# Notebook 11 — Lightweight CNN
if cnn_baseline_metrics_path.exists():
    df = pd.read_csv(cnn_baseline_metrics_path)
    row = df.iloc[0].to_dict()
    row["model"] = "Lightweight Mel CNN"
    comparison_rows.append(row)

# Notebook 12A — Multi-feature CNN
if multifeature_a_metrics_path.exists():
    df = pd.read_csv(multifeature_a_metrics_path)
    row = df.iloc[0].to_dict()
    row["model"] = "Multi-feature CNN (12A)"
    comparison_rows.append(row)

# Notebook 12B — Regularized Multi-feature CNN
if multifeature_b_metrics_path.exists():
    df = pd.read_csv(multifeature_b_metrics_path)
    row = df.iloc[0].to_dict()
    row["model"] = "Multi-feature CNN (12B)"
    comparison_rows.append(row)

# Notebook 13 — CNN-LSTM
comparison_rows.append({
    "model": "CNN-LSTM",
    "accuracy": accuracy_lstm,
    "balanced_accuracy": balanced_accuracy_lstm,
    "macro_precision": macro_precision_lstm,
    "macro_recall": macro_recall_lstm,
    "macro_f1": macro_f1_lstm,
    "weighted_f1": weighted_f1_lstm
})

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df[
    [
        "model",
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1"
    ]
]

print(comparison_df.to_string(index=False))

                  model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1
    Lightweight Mel CNN  0.716848           0.310168         0.213023      0.265858  0.212880     0.768721
Multi-feature CNN (12A)  0.855221           0.166417         0.142720      0.166417  0.153660     0.789662
Multi-feature CNN (12B)  0.856502           0.166667         0.142750      0.166667  0.153784     0.790299
               CNN-LSTM  0.813581           0.226085         0.134473      0.169564  0.146471     0.786735


In [52]:
ranking_df = comparison_df.sort_values(
    by=["macro_f1", "balanced_accuracy"],
    ascending=False
).reset_index(drop=True)

ranking_df.insert(
    0,
    "rank",
    np.arange(1, len(ranking_df) + 1)
)

print(ranking_df.to_string(index=False))

 rank                   model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  weighted_f1
    1     Lightweight Mel CNN  0.716848           0.310168         0.213023      0.265858  0.212880     0.768721
    2 Multi-feature CNN (12B)  0.856502           0.166667         0.142750      0.166667  0.153784     0.790299
    3 Multi-feature CNN (12A)  0.855221           0.166417         0.142720      0.166417  0.153660     0.789662
    4                CNN-LSTM  0.813581           0.226085         0.134473      0.169564  0.146471     0.786735


In [53]:
comparison_path = (
    CNN_LSTM_OUTPUT_DIR /
    "cnn_model_comparison.csv"
)

ranking_df.to_csv(
    comparison_path,
    index=False
)

print("Saved:", comparison_path)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/cnn_model_comparison.csv


In [54]:
best_model_row = ranking_df.iloc[0]

summary_lstm = {
    "experiment": "CNN-LSTM optional experiment",
    "best_model_by_macro_f1": best_model_row["model"],
    "best_macro_f1": float(best_model_row["macro_f1"]),
    "best_balanced_accuracy": float(
        best_model_row["balanced_accuracy"]
    ),
    "cnn_lstm_macro_f1": float(macro_f1_lstm),
    "cnn_lstm_balanced_accuracy": float(
        balanced_accuracy_lstm
    ),
    "cnn_lstm_accuracy": float(accuracy_lstm),
    "cnn_lstm_weighted_f1": float(weighted_f1_lstm),
    "patient_disjoint_evaluation": True,
    "training_augmentation_only": True
}

summary_path = (
    CNN_LSTM_OUTPUT_DIR /
    "cnn_lstm_experiment_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        summary_lstm,
        f,
        indent=2
    )

print("Saved:", summary_path)

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_lstm/cnn_lstm_experiment_summary.json


## CNN-LSTM Experiment — Interpretation

The CNN-LSTM experiment evaluates whether explicit temporal sequence
modeling improves respiratory disease classification compared with the
Lightweight Mel CNN and Multi-feature CNN models.

The CNN frontend extracts local time-frequency patterns from the
Log-Mel representation while preserving the temporal dimension.
The LSTM then models dependencies across the resulting sequence of
157 time frames.

The experiment uses the same patient-disjoint train, validation and
test split as the CNN baseline, making the comparison controlled.

Model performance is interpreted using balanced accuracy and macro F1
in addition to overall accuracy. This is important because the disease
distribution is highly imbalanced and overall accuracy can be dominated
by the majority COPD class.

The CNN-LSTM should therefore be considered beneficial only if it
provides a meaningful improvement in balanced accuracy and/or macro F1,
rather than simply increasing raw accuracy.

The CNN-LSTM is an optional experiment in this project. Its results
will be carried forward to the final model comparison in Notebook 16,
where the strongest models will be compared using the same evaluation
criteria.

### Experimental Conclusion

Among the evaluated deep learning approaches, the Lightweight Mel CNN
provides the strongest balanced disease classification performance,
achieving the highest balanced accuracy (0.3102) and macro F1 (0.2129).

The Multi-feature CNN models achieve substantially higher overall
accuracy (approximately 0.856), but their balanced accuracy and macro
F1 are considerably lower. This indicates that their high accuracy is
strongly influenced by the majority COPD class.

The CNN-LSTM achieves an accuracy of 0.8136, but its balanced accuracy
(0.2261) and macro F1 (0.1465) are lower than those of the Lightweight
Mel CNN. Therefore, explicit temporal sequence modeling did not improve
the primary disease-classification objectives in this experiment.

These results demonstrate that increasing feature complexity or model
complexity does not necessarily improve minority-class disease
discrimination. The Lightweight Mel CNN is therefore retained as the
strongest deep-learning baseline according to balanced accuracy and
macro F1.

Asthma and LRTI have no test samples in the patient-disjoint split
because of their extremely limited patient representation in the
dataset. Consequently, performance for these classes cannot be
estimated from this test split and this limitation must be considered
when interpreting the results.